# FightWatch — ưu tiên cảnh có người

Lọc clip theo người đủ rõ ở ít nhất hai trong ba frame, không đọc nhãn hay score để quyết định lọc. Giữ cảnh đông người bình thường, không gán nhiều người = đánh nhau. Cảnh không người/người quá nhỏ không thuộc cohort demo lần này; không tự gọi tất cả là giao thông. Thêm tối đa 15% mẫu âm một người từ train để tránh báo nhầm một người vận động.

Phạm vi validation thay đổi nên phải tính lại các baseline trên cùng subset; vẫn báo kết quả model mới trên toàn bộ validation gốc. Không loại clip chỉ vì model dự đoán sai. Không tự duyệt checkpoint demo. Budget script 12 phút.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, shutil, json, zipfile, time, threading
ROOT = Path('/content/drive/MyDrive/SENTINEL')
SOURCE = ROOT / 'artifacts/fight_pilot_x3d_v1'
OUTPUT = ROOT / 'artifacts/fight_people_focus_v1'
CACHE = ROOT / 'features/fight_ablation_v2'
DATA = Path('/content/fightwatch-data/UBI_FIGHTS')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-code')
URL = 'https://github.com/TranDuon/anomaly-behavior-detector-system.git'
BUDGET_MINUTES = 12
for directory in (OUTPUT, CACHE, DATA, REPO.parent):
    directory.mkdir(parents=True, exist_ok=True)
assert (SOURCE / 'manifest.json').exists() and (SOURCE / 'report.json').exists(), 'Thiếu artifact pilot gốc'
if (REPO / '.git').exists():
    remote = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote.removesuffix('.git') == URL.removesuffix('.git'), 'Sai remote repo Drive'
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', URL, str(REPO)], check=True)
shutil.copytree(REPO, LOCAL, dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__', 'models', 'uploads', 'logs', '.env', '.pytest_cache'))
os.chdir(LOCAL)
revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(OUTPUT / 'code_revision.txt').write_text(revision)
print('REVISION', revision)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Chọn GPU trước khi chạy tiếp'
print('GPU', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_fight_people.py', 'tests/test_fight_runtime.py', '-q', '-p', 'no:cacheprovider'], check=True)


In [ ]:
manifest = json.loads((SOURCE / 'manifest.json').read_text())
selected = sorted({row['key'] for row in manifest['rows']})
archive_path = ROOT / 'datasets/fight/UBI_FIGHTS.zip'
assert archive_path.stat().st_size == 8_117_112_419, 'Archive gốc không khớp kích thước'
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    test_name = next(name for name in names if name.endswith('/test_videos.csv') or name == 'test_videos.csv')
    official_test = set(archive.read(test_name).decode().splitlines())
    assert not set(selected) & official_test
    videos = {Path(name).stem: name for name in names if Path(name).suffix.lower() in {'.mp4', '.avi'} and '/videos/' in '/' + name}
    annotations = {Path(name).stem: name for name in names if '/annotation/' in '/' + name and name.endswith('.csv')}
    for index, key in enumerate(selected, 1):
        for member in (videos[key], annotations[key]):
            parts = Path(member).parts
            marker = 'videos' if 'videos' in parts else 'annotation'
            destination = DATA.joinpath(*parts[parts.index(marker):])
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists() or destination.stat().st_size != archive.getinfo(member).file_size:
                temporary = destination.with_suffix(destination.suffix + '.partial')
                with archive.open(member) as source, temporary.open('wb') as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(destination)
        if index % 8 == 0:
            print('Prepared', index, '/', len(selected), flush=True)
    (DATA / 'test_videos.csv').write_bytes(archive.read(test_name))
print('Đã chuẩn bị đúng các video của split cũ; không giải nén toàn dataset.')


In [ ]:
command = [sys.executable, '-u', '-m', 'scripts.fight_people', '--source-run', str(SOURCE), '--ablation-run', str(ROOT / 'artifacts/fight_ablation_v2'), '--dataset-root', str(DATA), '--output-dir', str(OUTPUT), '--cache-dir', str(CACHE), '--budget-minutes', str(BUDGET_MINUTES)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def stop_over_budget():
    if process.poll() is None:
        process.terminate()
        threading.Timer(10, lambda: process.kill() if process.poll() is None else None).start()
watchdog = threading.Timer(BUDGET_MINUTES * 60 + 120, stop_over_budget)
watchdog.start()
try:
    with (OUTPUT / 'run_log.txt').open('a') as log:
        log.write(f'\nREVISION {revision} | {time.ctime()}\n')
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
    return_code = process.wait()
finally:
    watchdog.cancel()
assert return_code == 0, 'Phiên đối chứng dừng sớm; xem run_log.txt/comparison.json, không tự train tiếp'


In [ ]:
import pandas as pd
from IPython.display import display, Image
report = json.loads((OUTPUT / 'report.json').read_text())
print('Decision:', report['decision'], '|', report.get('reason'))
print('Train:', report.get('train_clips'), '| People validation:', report.get('validation_clips'), '| Original validation:', report.get('original_validation_clips'))
coverage_rows = []
for split, labels in report.get('coverage', {}).items():
    for label, values in labels.items():
        coverage_rows.append({'split': split, 'label': label, 'source_clips': values['source_clips'], **values['domains']})
display(pd.DataFrame(coverage_rows).fillna(0))
summary = []
for name, baseline in report.get('baselines_on_same_cohort', {}).items():
    values = baseline['people_scope']
    summary.append({'model': name, **values['validation'], 'recall_at_precision75': values['best_recall_at_precision75']})
if 'validation' in report:
    summary.append({'model': 'people_focus_v1', **report['validation'], 'recall_at_precision75': report['best_recall_at_precision75']})
if summary:
    table = pd.DataFrame(summary).set_index('model')
    display(table)
    (OUTPUT / 'summary.html').write_text(table.to_html())
print('Original full validation at the same people threshold:', report.get('original_scope_at_people_threshold'))
print('Subset đổi phạm vi, không so thẳng số headline với 109 clip cũ. Model vẫn là pilot chưa duyệt.')
for entry in json.loads((OUTPUT / 'gallery.json').read_text()):
    print(entry['domain'], entry['video'], entry['start_seconds'], 'label=', entry['label'], 'people=', entry['salient_people_counts'])
    display(Image(filename=str(OUTPUT / entry['image'])))
print('Saved:', OUTPUT)
